In [1]:
from pathlib import Path
import pandas as pd
import torch
import matplotlib.pyplot as plt
import xarray as xr
import yaml
import geopandas as gpd
import numpy as np
import pickle
from neuralhydrology.evaluation.metrics import calculate_all_metrics
from neuralhydrology.evaluation.metrics import calculate_metrics
from neuralhydrology.evaluation.metrics import missed_peaks
import neuralhydrology

In [2]:
# ------------------- Paths -------------------
RUN_DIR = Path("../runs") #Uruguay
# RUN_DIR = Path("../extending_caravan/runs") #USA

# Ensemble metrics

In [26]:
# URUGUAY

# run_pattern = "precip_prcp_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_seed_*"
# run_pattern = "precip_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_seed_*" 
# run_pattern = "precip_prcp_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*" 
run_pattern = "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*" 
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*" 

matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/validation/model_epoch030/validation_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 8 runs: ['precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111_2202_234508', 'precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_222_2202_235105', 'precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_333_2202_235701', 'precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_444_2302_000258', 'precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_555_2302_114942', 'precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_666_2302_115552', 'precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_777_2302_120201', 'precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_888_2302_120808']


In [27]:
# USA

# run_pattern = "154_camels_30_epochs_seq_270_hidden_256_dropout_04_fb_05_seed111_1103_063301" # CAMELS
# run_pattern = "caravan_154_precip_seq_270_30_epochs_hidden_256_dropout_04_fb_05_seed111_0103_211409" # CARAVAN
# run_pattern = "precip_chirps_precipitation_0503_093821" # CHIRPS
# run_pattern = "precip_mswep_precipitation_0503_110038" # MSWEP
# run_pattern = "precip_total_precipitation_sum_chirps_precipitation_0303_072447" # CARAVAN + CHIRPS
# run_pattern = "precip_total_precipitation_sum_mswep_precipitation_0303_084843" # CARAVAN + MSWEP
# run_pattern = "precip_chirps_precipitation_mswep_precipitation_0303_101255" # CHIRPS + MSWEP
# run_pattern = "precip_total_precipitation_sum_chirps_precipitation_mswep_precipitation_0303_113714" # CARAVAN + CHIRPS + MSWEP

# matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/validation/model_epoch030/validation_results.p"))
# print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

In [28]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['QObs_mm_d_sim'].values #['QObs(mm/d)_sim'].values 
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy()
    xr_ensemble['QObs_mm_d_sim'].values[:] = mean_sim #['QObs(mm/d)_sim'].values[:] = mean_sim #
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

ensemble_data

{'CAMELS_UY_10': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:        (date: 3652, time_step: 1)
   Coordinates:
     * date           (date) datetime64[ns] 29kB 1989-10-01 ... 1999-09-30
     * time_step      (time_step) int64 8B 0
   Data variables:
       QObs_mm_d_obs  (date, time_step) float32 15kB 0.09307 0.1167 ... 0.09939
       QObs_mm_d_sim  (date, time_step) float32 15kB 0.5345 0.4904 ... 0.5176}},
 'CAMELS_UY_11': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:        (date: 3652, time_step: 1)
   Coordinates:
     * date           (date) datetime64[ns] 29kB 1989-10-01 ... 1999-09-30
     * time_step      (time_step) int64 8B 0
   Data variables:
       QObs_mm_d_obs  (date, time_step) float32 15kB 0.263 0.2622 ... 0.4166 0.4053
       QObs_mm_d_sim  (date, time_step) float32 15kB 0.3428 0.3196 ... 0.4303}},
 'CAMELS_UY_15': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:        (date: 3652, time_step: 1)
   Coordinates:
     * date           (d

In [18]:
save_name = run_pattern.split("_seed_")[0]
hydrographs_dir = Path(f"./hydrographs/{save_name}")
hydrographs_dir.mkdir(parents=True, exist_ok=True)

def compute_nse(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs, sim = obs[mask], sim[mask]
    return 1 - np.sum((obs - sim) ** 2) / np.sum((obs - np.mean(obs)) ** 2)

for catchment, content in ensemble_data.items():
    ds = content["1D"]["xr"]

    time = ds["date"].values
    obs  = ds["QObs_mm_d_obs"].values.flatten()
    sim  = ds["QObs_mm_d_sim"].values.flatten()

    nse = compute_nse(obs, sim)

    plt.figure(figsize=(12, 5))
    plt.plot(time, obs, label="Observed", alpha=0.6)
    plt.plot(time, sim, label="Simulated", alpha=0.6)
    plt.xlabel("Date")
    plt.ylabel("Streamflow (mm/day)")
    plt.title(f"Validation - {catchment} - NSE: {nse:.3f}")
    plt.legend()
    plt.grid(True, alpha=0.35)
    plt.tight_layout()

    out_path = hydrographs_dir / f"hydrograph_{catchment}.png"
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"Saved {catchment} | NSE: {nse:.3f}")

Saved CAMELS_UY_10 | NSE: 0.498
Saved CAMELS_UY_11 | NSE: 0.160
Saved CAMELS_UY_15 | NSE: 0.810
Saved CAMELS_UY_16 | NSE: 0.324
Saved CAMELS_UY_2 | NSE: 0.661
Saved CAMELS_UY_3 | NSE: 0.823
Saved CAMELS_UY_5 | NSE: 0.795
Saved CAMELS_UY_6 | NSE: 0.873
Saved CAMELS_UY_7 | NSE: 0.599
Saved CAMELS_UY_8 | NSE: 0.703
Saved CAMELS_UY_9 | NSE: 0.845


In [29]:
save_name = "slides_version"
hydrographs_dir = Path(f"./hydrographs/{save_name}")
hydrographs_dir.mkdir(parents=True, exist_ok=True)

def compute_nse(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs, sim = obs[mask], sim[mask]
    return 1 - np.sum((obs - sim) ** 2) / np.sum((obs - np.mean(obs)) ** 2)

# Selected basins and their y-axis limits
selected_basins = {
    "CAMELS_UY_3": (0, 55),
}

for catchment, ylim in selected_basins.items():
    ds = ensemble_data[catchment]["1D"]["xr"]

    time = ds["date"].values
    obs  = ds["QObs_mm_d_obs"].values.flatten()
    sim  = ds["QObs_mm_d_sim"].values.flatten()

    nse = compute_nse(obs, sim)

    plt.figure(figsize=(12, 5))
    plt.plot(time, obs, label="Observed", alpha=0.6)
    plt.plot(time, sim, label="Simulated", alpha=0.6)
    plt.xlabel("Date")
    plt.ylabel("Streamflow (mm/day)")
    plt.title(f"Validation - {catchment} - NSE: {nse:.3f}")
    plt.ylim(ylim)
    plt.legend()
    plt.grid(True, alpha=0.35)
    plt.tight_layout()

    out_path = hydrographs_dir / f"chirps_mswep_gauges_hydrograph_{catchment}.png"
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"Saved {catchment} | NSE: {nse:.3f}")

Saved CAMELS_UY_3 | NSE: 0.823
